# Link-level simulation

> Every bit of a transmission: uncoded (`BitLevelLink`) or 5G NR coded with Sionna PHY (`SionnaLink`), and the channels built on them.

In [ ]:
#| default_exp link_level

In [ ]:
#| hide
from nbdev.showdoc import *

Link-level simulation in the sense of [Sionna PHY](https://nvlabs.github.io/sionna/phy/index.html): what happens
to the bits of one transmission.

1. The payload becomes bits (a `Codec`, see `dsp.source_coding`).
2. The bits become symbols (a `Modulation`, see `dsp.modulation`).
3. The symbols go through the channel, `y = h √P x + n` (see `dsp.signal`), with `h` and `P` from the channel's
   fading and power control (see `fading` and `power_control`).
4. The receiver detects the bits and decodes them, so it gets the payload with its bit errors.

`BitLevelLink` puts these together as a block (see `block`), and `LinkLevelChannel` is the radio channel whose
receiver it is. `SionnaLink` and `SionnaLinkChannel`, further down, do the same for 5G NR, with channel coding. It is slower than `SystemLevelChannel`, which
skips the bits, and it takes the same fading and power control: the same scenario runs at either level.

The pieces also work on their own, vectorized, without a `Network` (see the end of this page).

In [ ]:
#| export
from __future__ import annotations

import math
from collections import defaultdict
from typing import Sequence

import numpy as np

from comm_core.block import Block
from comm_core.core import Message, Transmission, TransmissionResult
from comm_core.dsp.modulation import BPSK, Modulation
from comm_core.dsp.signal import apply_channel
from comm_core.dsp.source_coding import Codec, IndexCodec
from comm_core.fading import Fading, effective_coefficient
from comm_core.interference import SINR
from comm_core.link_adaptation import NRMCSTable
from comm_core.phy_abstraction import SionnaErrorModel
from comm_core.power_control import PowerControl
from comm_core.radio import RadioChannel
from comm_core.topology import Link

## Link-level channel

`LinkLevelChannel` simulates every bit of every transmission. It is a `RadioChannel` (see `radio`) whose receiver is
`BitLevelLink`, the pieces above: the bits of each transmission go through `h √P x + n`, with the interference
added to the noise.

* **Delivery.** A sent message is delivered with its bit errors (`success=True`): the payload the receiver gets may
  differ from what was sent.
* **No CRC by default.** Pass `drop_corrupted=True` to drop corrupted packets instead, like a receiver that checks a
  CRC.
* **Outages** are lost, as at the system level.

`size_bits` is what the codec produced. `error_rate` is the transmission's bit error rate, and `metadata` adds
`bit_errors`.

In [ ]:
#| export
class BitLevelLink(Block):
    "Every bit of each transmission: payload -> bits -> symbols -> h √P x + n (interference as noise) -> detected bits -> payload."

    outputs = ('success', 'payload', 'error_rate', 'num_bits', 'num_symbols', 'info')

    def __init__(
        self,
        codec: Codec | None = None, # Payload <-> bits; None: `IndexCodec(256)`
        modulation: Modulation | None = None, # Bits <-> symbols; None: `BPSK()`
        drop_corrupted: bool = False, # Lose the packets with bit errors (a CRC) instead of delivering them
    ):
        self.codec = codec if codec is not None else IndexCodec()
        self.modulation = modulation if modulation is not None else BPSK()
        self.drop_corrupted = drop_corrupted

    def call(
        self,
        transmission, # The transmissions (their payloads)
        h: np.ndarray, # h of each at each subcarrier, (n, K) (several: their mean power)
        power: np.ndarray, # Their transmit powers
        noise_power: float, # Noise power at the receivers
        interference: np.ndarray | None = None, # Power of the other transmissions at each receiver (added to the noise)
        active: np.ndarray | None = None, # Whether each one is sent (None: all); the others only count their bits
    ) -> tuple:
        "(success, payload, error_rate, num_bits, num_symbols, info) of each transmission."
        n = len(transmission)
        success, payload, error = np.zeros(n, dtype=bool), [None] * n, np.ones(n)
        num_bits, num_symbols, info = np.zeros(n, np.int64), np.zeros(n, np.int64), [{} for _ in range(n)]
        per_symbol = self.modulation.bits_per_symbol
        for k, t in enumerate(transmission):
            bits = np.asarray(self.codec.encode(t.message.payload))
            num_bits[k], num_symbols[k] = bits.size, -(-int(bits.size) // per_symbol)
            if active is not None and not active[k]:
                continue
            if np.isinf(h[k]).any():                     # same place: a perfect link
                success[k], payload[k], error[k], num_symbols[k], info[k] = True, t.message.payload, 0.0, 0, {'bit_errors': 0}
                continue
            coefficient = effective_coefficient(h[k])
            noise = noise_power + (float(interference[k]) if interference is not None else 0.0)   # interference as noise
            if not (float(power[k]) * abs(coefficient) ** 2 > 0 and math.isfinite(noise)):
                continue                                 # nothing to detect: no signal, or drowned (an interferer at the receiver's place)
            last = int(bits.shape[-1])
            padded = np.pad(bits, [(0, 0)] * (bits.ndim - 1) + [(0, -last % per_symbol)])
            symbols = self.modulation.modulate(padded)
            received = apply_channel(symbols, coefficient, float(power[k]), noise, self.rng)
            detected = np.asarray(self.modulation.demodulate(received, coefficient))[..., :last]
            bit_errors = int((detected != bits).sum())
            success[k] = not (self.drop_corrupted and bit_errors)
            payload[k] = self.codec.decode(detected) if success[k] else None
            error[k], num_symbols[k], info[k] = bit_errors / max(int(bits.size), 1), int(np.size(symbols)), {'bit_errors': bit_errors}
        return success, payload, error, num_bits, num_symbols, info


class LinkLevelChannel(RadioChannel):
    "Every bit simulated: payload -> bits -> symbols -> h √P x + n -> detected bits -> payload (with its bit errors)."

    def __init__(
        self,
        codec: Codec | None = None, # Payload <-> bits; None: `IndexCodec(256)`
        modulation: Modulation | None = None, # Bits <-> symbols; None: `BPSK()`
        fading: Fading | None = None, # The coefficient h of a transmission; None: h = 1
        power_control: PowerControl | None = None, # Transmit power and outages; None: the link's tx_power_w
        noise_power: float = 1.0, # Noise power at the receiver (same unit as the power)
        drop_corrupted: bool = False, # Lose the packets with bit errors (a CRC) instead of delivering them
        seed: int | None = None, # Seed of the noise and the fading
    ):
        self.link_level = BitLevelLink(codec, modulation, drop_corrupted)
        super().__init__(fading, power_control, noise_power, seed)

    @property
    def codec(self) -> Codec:
        return self.link_level.codec

    @property
    def modulation(self) -> Modulation:
        return self.link_level.modulation

    def receiver(self) -> list:
        "The bits: `BitLevelLink`."
        return [self.link_level]

In [ ]:
show_doc(BitLevelLink.call)

### Tests

In [ ]:
from comm_core.core import Message
from comm_core.dsp.source_coding import UniformQuantizer, to_bits
from comm_core.phy_abstraction import bpsk_bit_error_rate, packet_error_rate
from comm_core.fading import NoFading, RayleighFading
from comm_core.power_control import ChannelInversion, MessagePower
from comm_core.system_level import SystemLevelChannel

In [ ]:
def send(ch, payload, link=Link('a', 'b'), n=1, **metadata):
    t = Transmission(message=Message('a', 'b', payload, metadata=metadata), sender='a', receiver='b', start_time=0.0)
    return [ch.transmit(t, link) for _ in range(n)]

idx = np.arange(49) * 5                                          # 49 codebook indices
r, = send(LinkLevelChannel(noise_power=1e-6, seed=0), idx)        # 60 dB: no errors
assert r.success and (r.payload == idx).all() and r.size_bits == 49 * 8 and r.metadata['bit_errors'] == 0
assert abs(r.snr_db - 60) < 1e-9 and r.energy == 49 * 8 and r.latency == 0
# 0 dB: about 8% of the bits are wrong, and the receiver gets the corrupted indices
rs = send(LinkLevelChannel(noise_power=1.0, seed=0), idx, n=50)
ber = np.mean([r.error_rate for r in rs])
assert all(r.success for r in rs) and abs(ber - bpsk_bit_error_rate(1.0)) < 0.01
assert not (rs[0].payload == idx).all()
# a CRC: corrupted packets are lost
assert not any(r.success for r in send(LinkLevelChannel(noise_power=1.0, drop_corrupted=True, seed=0), idx, n=20))
# channel inversion: the weak link (|h|^2 = 0.01) would need 1000 > 10: outage, nothing sent
for h, sent in ((1.2, True), (0.1, False)):
    r, = send(LinkLevelChannel(fading=NoFading(h), power_control=ChannelInversion(10, max_power=10), seed=0), idx)
    assert r.success == sent and r.size_bits == (392 if sent else 0) and r.metadata.get('outage', False) != sent
    if sent:
        assert abs(r.snr_db - 10) < 1e-6
# the sender chooses the power
ch = LinkLevelChannel(power_control=MessagePower(), seed=0)
assert not send(ch, idx)[0].success and send(ch, idx, tx_power=100.0)[0].metadata['tx_power'] == 100.0
# a finite bandwidth: latency = symbols / bandwidth, energy = power x latency
r, = send(LinkLevelChannel(noise_power=1e-6), idx, link=Link('a', 'b', bandwidth_hz=1e6, tx_power_w=0.5))
assert abs(r.latency - 392e-6) < 1e-15 and abs(r.energy - 0.5 * 392e-6) < 1e-15
# Rayleigh fading: varying SNR, reproducible with reset(seed)
ch = LinkLevelChannel(fading=RayleighFading(), noise_power=0.1)
ch.reset(seed=1); first = [r.snr_db for r in send(ch, idx, n=5)]
ch.reset(seed=1); assert [r.snr_db for r in send(ch, idx, n=5)] == first and len(set(first)) == 5
# real numbers: quantized to 8 bits
r, = send(LinkLevelChannel(codec=UniformQuantizer(8, -1, 1), noise_power=1e-6), np.array([0.25, -0.5]))
assert r.size_bits == 16 and np.allclose(r.payload, [0.25, -0.5], atol=1 / 255)

In [ ]:
from comm_core.backend import PipelineBackend
from comm_core.power_control import FixedPower
from comm_core.radio import Airtime

In [ ]:
# the link level is its blocks: the same results from them in a pipeline, and a batch is one at a time
ch = LinkLevelChannel(fading=RayleighFading(), noise_power=0.5)
blocks = PipelineBackend([RayleighFading(), FixedPower(), BitLevelLink(), Airtime()], noise_power=0.5)
assert [type(b).__name__ for b in ch.pipeline()] == [type(b).__name__ for b in blocks.pipeline]
t = Transmission(message=Message('a', 'b', idx), sender='a', receiver='b', start_time=0.0)
ch.reset(seed=2); one = [ch.transmit(t, Link('a', 'b')).error_rate for _ in range(20)]
ch.reset(seed=2); batch = [r.error_rate for r in ch.transmit_batch([(t, Link('a', 'b'))] * 20)]
assert one == batch and len(set(one)) > 1

In [ ]:
from comm_core.dsp.modulation import SionnaModulation

In [ ]:
#| sionna
# 16-QAM through the link level: a quarter of the symbols, and the errors of its SNR
ch = LinkLevelChannel(modulation=SionnaModulation('qam', 4), noise_power=1e-3, seed=0)
r, = send(ch, idx)
assert r.success and (r.payload == idx).all() and r.metadata['bit_errors'] == 0 and r.energy == 392 / 4
rs = send(LinkLevelChannel(modulation=SionnaModulation('qam', 4), noise_power=10 ** -1.2, seed=0), idx, n=100)
ber = np.mean([r.error_rate for r in rs])
assert 0.02 < ber < 0.04                                        # 12 dB: about 0.028

The two levels agree. With a CRC, the link-level channel loses packets as often as the system level's PHY abstraction
says it does:

In [ ]:
idx = np.arange(8) * 30                                          # 64 bits
for snr_db in (4, 6, 8):
    noise = 10 ** (-snr_db / 10)
    link_level = LinkLevelChannel(noise_power=noise, drop_corrupted=True, seed=0)
    lost = 1 - np.mean([r.success for r in send(link_level, idx, n=1000)])
    expected = packet_error_rate(snr_db, 64)
    assert abs(lost - expected) < 0.04, (snr_db, lost, expected)

With interference (see `InterferenceBackend`), the link level adds it to the noise of every bit, and gives the BER of the SINR:

In [ ]:
from comm_core.fading import RayTracedFading
from comm_core.radio import InterferenceBackend

In [ ]:
xs = np.arange(5)
d = np.abs(np.subtract.outer(xs, xs)).astype(float)
table = RayTracedFading(np.where(d > 0, 1e-3 / np.maximum(d, 1) ** 1.5, 0), [(x, 0) for x in xs],
                        node_positions={'a': (0, 0), 'b': (1, 0), 'c': (2, 0), 'd': (3, 0)})
backend = InterferenceBackend(LinkLevelChannel(fading=table, noise_power=1e-10, seed=0))
pair = lambda s, r: (Transmission(Message(s, r, np.arange(256) % 256), s, r, 0.0), Link(s, r))
r, _ = backend.transmit_all([pair('a', 'b'), pair('c', 'd')])        # c is as close to b as a: SINR about 0 dB
sinr = 1e-6 / (1e-10 + 1e-6)
assert abs(r.snr_db - 10 * np.log10(sinr)) < 1e-9 and abs(r.metadata['interference_power'] - 1e-6) < 1e-18
assert abs(r.error_rate - bpsk_bit_error_rate(sinr)) < 0.02 and r.success

## Batched, without a network

Training code can apply the same steps to a whole batch of links at once, on numpy arrays or torch tensors, e.g.
`B·T` samples × `n` neighbours, each sending 49 codebook indices, with channel inversion. Here `csi` holds the
coefficients `(B·T, n, 1)`, and links in an outage get the null index `codebook_size`:

```python
codec, bpsk = IndexCodec(256), BPSK()
power, send = ChannelInversion(target_snr_db=10, max_power=10).allocate(csi.abs() ** 2, noise_power)
x = bpsk.modulate(codec.encode(msg_indices[:, None, :]))           # (B·T, 1, 392), the same message to every neighbour
y = apply_channel(x, csi, power, noise_power)                        # (B·T, n, 392)
received = codec.decode(bpsk.demodulate(y, csi))                   # (B·T, n, 49)
received = torch.where(send, received, codec.codebook_size)        # nothing received in an outage
```

In [ ]:
#| hide
def batch(xp, msg_indices, csi, noise_power=1.0, rng=None):
    codec, bpsk = IndexCodec(256), BPSK()
    power, send = ChannelInversion(target_snr_db=10, max_power=10).allocate(abs(csi) ** 2, noise_power)
    x = bpsk.modulate(codec.encode(msg_indices[:, None, :]))
    y = apply_channel(x, csi, power, noise_power, rng)
    received = codec.decode(bpsk.demodulate(y, csi))
    return xp.where(send, received, codec.codebook_size), send

rng = np.random.default_rng(0)
msg = rng.integers(0, 256, (64, 49))
csi = (rng.standard_normal((64, 3, 1)) + 1j * rng.standard_normal((64, 3, 1))) / np.sqrt(2)
received, sent = batch(np, msg, csi, rng=rng)
assert received.shape == (64, 3, 49) and ((received == 256) == ~sent).all()
ber = (to_bits(np.where(sent, received, 0), 8) != to_bits(np.broadcast_to(msg[:, None], received.shape), 8))[sent[..., 0]].mean()
assert abs(ber - bpsk_bit_error_rate(10.0)) < 1e-3                # every sent link at 10 dB
try:
    import torch
except ImportError:
    torch = None
if torch is not None:                                               # the same on torch tensors
    g = torch.Generator().manual_seed(0)
    received_t, sent_t = batch(torch, torch.as_tensor(msg), torch.as_tensor(csi), rng=g)
    assert received_t.shape == (64, 3, 49) and torch.equal(sent_t, torch.as_tensor(sent))
    assert ((received_t == 256) == ~sent_t).all()

## 5G NR, with Sionna PHY

The link level of 5G NR with [Sionna PHY](https://nvlabs.github.io/sionna/phy/index.html): every bit of every
transmission, coded and decoded the way an NR receiver does it.

1. The payload becomes bits (a `Codec`, e.g. `IndexCodec` for codebook indices).
2. **Coding:** the bits become an NR transport block, with a CRC, LDPC coding and rate matching (`TBEncoder`), at
   the MCS of the transmission.
3. **Modulation:** QAM (`Mapper`).
4. **The channel:** the symbols go out on resource elements, filling the subcarriers in turn. Each sees the
   channel of its subcarrier: `y = h_k √P x + n`, where the noise includes any interference.
5. **The receiver:** it equalizes each resource element (one tap, knowing `h_k`), computes soft bits (`Demapper`,
   LLRs), and decodes the LDPC code (`TBDecoder`).

The CRC decides delivery, as in a real receiver: a transport block that fails its CRC is lost.

`SionnaLink` is this chain as a block (see `block`), and `SionnaLinkChannel` is the `RadioChannel` whose receiver
it is, like `SystemLevelChannel` and `LinkLevelChannel` (above). It takes the same
fadings (e.g. `RayTracedPathsFading` with subcarriers), power control, `InterferenceBackend` and protocols. It needs
Sionna 2 (PyTorch): `pip install 'comm-core[sionna]'`.

**The MCS:** a fixed `mcs_index`, or link adaptation (the default). Link adaptation picks the highest MCS that
Sionna SYS (`SionnaErrorModel`) predicts to meet `bler_target` at the transmission's SINR. The transmitter is assumed
to know the channel.

**The results** add `mcs`, `resource_elements`, `tb_size` (the NR transport block, padded to an allowed size) and
`bit_errors` to their metadata. `error_rate` is the bit error rate of the decoded payload.

**MCSs:** those with code rates below 1/5 (MCS 0–2 of table 1) are not available. Sionna's LDPC encoder
doesn't support them (NR reaches them by repetition), so link adaptation chooses from the others.

**Cost:** LDPC decoding is heavy, around 25 ms per 1000-bit message on a CPU, much less on a GPU. Use it to check the
system level (`SionnaErrorModel`), or where the bits themselves matter. For training, `SystemLevelChannel` with
`SionnaErrorModel` is the fast equivalent.

In [ ]:
#| export
MIN_LDPC_CODERATE = 0.2                                  # Sionna's 5G LDPC encoder: rates below 1/5 are not supported


class SionnaLink(Block):
    "5G NR bits with Sionna PHY, as a block: LDPC-coded with a CRC, QAM, through h at each subcarrier, equalized and decoded."

    outputs = ('success', 'payload', 'error_rate', 'num_bits', 'num_symbols', 'info')

    def __init__(
        self,
        codec: Codec | None = None, # Payload <-> bits; None: `IndexCodec(256)`
        mcs_index: int | None = None, # A fixed MCS; None: link adaptation (see `SionnaErrorModel`)
        bler_target: float = 0.1, # Link adaptation's target error rate
        mcs_table_index: int = 1, # 5G NR MCS table: 1 (up to 64-QAM) or 2 (up to 256-QAM)
        mcs_category: int = 0, # 0: PUSCH (uplink), 1: PDSCH (downlink)
        num_bp_iter: int = 20, # LDPC decoder iterations
        device: str | None = None, # Torch device; None: Sionna's default (a GPU if there is one)
        precision: str = 'single', # 'single' or 'double'
    ):
        import torch
        self._torch = torch
        self.codec = codec if codec is not None else IndexCodec()
        supported = NRMCSTable(mcs_table_index, mcs_category, min_coderate=MIN_LDPC_CODERATE, device=device).indices
        if mcs_index is not None and mcs_index not in supported:
            raise ValueError(f"MCS {mcs_index}: not one Sionna PHY can encode (code rates below 1/5 are not supported); "
                             f"one of {supported}")
        self.adaptation = SionnaErrorModel(mcs_index=mcs_index, bler_target=bler_target, mcs_table_index=mcs_table_index,
                                           mcs_category=mcs_category, device=device, precision=precision,
                                           mcs_indices=supported)
        self.device, self.precision = self.adaptation.device, precision
        self.real = torch.float64 if precision == 'double' else torch.float32
        self.complex = torch.complex128 if precision == 'double' else torch.complex64
        self.mcs_table_index, self.mcs_category, self.num_bp_iter = mcs_table_index, mcs_category, num_bp_iter
        self._links = {}                                  # (mcs, bits) -> the coding chain, built once

    def _link(self, mcs: int, num_bits: int) -> tuple:
        "(bits per symbol, resource elements, encoder, decoder, mapper, demapper) of an MCS and a payload size."
        if (mcs, num_bits) not in self._links:
            import contextlib, io
            torch = self._torch
            from sionna.phy.mapping import Demapper, Mapper
            from sionna.phy.nr import TBDecoder, TBEncoder
            q, r = self.adaptation.table.mcs(mcs)
            num_re, tb_size = self.adaptation.allocation(num_bits, mcs)     # the same allocation as the system level
            with contextlib.redirect_stdout(io.StringIO()):
                encoder = TBEncoder(target_tb_size=tb_size, num_coded_bits=num_re * q, target_coderate=r,
                                    num_bits_per_symbol=q, channel_type='PUSCH' if self.mcs_category == 0 else 'PDSCH',
                                    precision=self.precision, device=self.device)
            self._links[mcs, num_bits] = (q, num_re, encoder,
                                         TBDecoder(encoder, num_bp_iter=self.num_bp_iter, precision=self.precision, device=self.device),
                                         Mapper('qam', q, precision=self.precision, device=self.device),
                                         Demapper('app', 'qam', q, precision=self.precision, device=self.device))
        return self._links[mcs, num_bits]

    def call(
        self,
        transmission, # The transmissions (their payloads)
        h: np.ndarray, # h of each at each subcarrier, (n, K)
        power: np.ndarray, # Their transmit powers
        sinr: np.ndarray, # Their SINR at each subcarrier, (n, K): link adaptation's (the transmitter's channel knowledge)
        noise_power: float, # Noise power at the receivers, per resource element
        interference: np.ndarray | None = None, # Power of the other transmissions at each receiver (added to the noise)
        group: Sequence | None = None, # The broadcast of each (one MCS for its copies), None for a unicast
        active: np.ndarray | None = None, # Whether each one is sent (None: all); the others only count their bits
    ) -> tuple:
        "(success, payload, error_rate, num_bits, num_symbols, info): the ones with the same MCS and size coded and decoded together."
        torch = self._torch
        n = len(transmission)
        success, payload, error = np.zeros(n, dtype=bool), [None] * n, np.ones(n)
        num_bits, num_symbols, info = np.zeros(n, np.int64), np.zeros(n, np.int64), [{} for _ in range(n)]
        bits, todo = {}, []
        for k, t in enumerate(transmission):
            bits[k] = np.asarray(self.codec.encode(t.message.payload))
            num_bits[k] = num_symbols[k] = bits[k].size
            if active is not None and not active[k]:
                continue
            if np.isinf(h[k]).any():                      # same place: a perfect link
                success[k], payload[k], error[k], num_symbols[k], info[k] = True, t.message.payload, 0.0, 0, {'bit_errors': 0}
            else:
                todo.append(k)
        if not todo:
            return success, payload, error, num_bits, num_symbols, info
        interference = np.zeros(n) if interference is None else np.asarray(interference, float)
        # the MCS of each transmission (fixed, or link adaptation from its SINR at each subcarrier; one per broadcast)
        _, _, rated = self.adaptation([sinr[k] for k in todo], np.array([bits[k].size for k in todo]),
                                      group=None if group is None else [group[k] for k in todo])
        batches = defaultdict(list)
        for k, i in zip(todo, rated):
            batches[i['mcs'], int(bits[k].size)].append(k)
        for (mcs, size), ks in batches.items():
            q, num_re, encoder, decoder, mapper, demapper = self._link(mcs, size)
            padded = np.zeros((len(ks), encoder.k))      # the payload, zero-padded to the transport block
            for row, k in enumerate(ks):
                padded[row, :size] = bits[k].reshape(-1)
            b = torch.as_tensor(padded, dtype=self.real, device=self.device)
            x = mapper(encoder(b))                        # (batch, resource elements)
            # the channel of each resource element: its subcarrier's, filling the subcarriers in turn
            hre = np.stack([np.resize(h[k], num_re) * math.sqrt(float(power[k])) for k in ks])
            hre = torch.as_tensor(hre, dtype=self.complex, device=self.device)
            var = torch.as_tensor([noise_power + interference[k] for k in ks], dtype=self.real, device=self.device)[:, None]
            gen = torch.Generator(device=self.device).manual_seed(int(self.rng.integers(2 ** 62)))
            noise = torch.complex(torch.randn(x.shape, generator=gen, device=self.device, dtype=self.real),
                                  torch.randn(x.shape, generator=gen, device=self.device, dtype=self.real))
            y = hre * x + noise.to(self.complex) * torch.sqrt(var / 2)
            gain = (hre.abs() ** 2).clamp_min(1e-30)
            llr = demapper(y / hre, var / gain)           # one-tap equalizer: the noise of each resource element
            b_hat, crc = decoder(llr)
            b_hat, crc = b_hat.cpu().numpy().round().astype(np.int64), crc.cpu().numpy()
            for row, k in enumerate(ks):
                decoded = b_hat[row, :size].reshape(bits[k].shape)
                errors = int((decoded != bits[k]).sum())
                success[k] = bool(crc[row])
                payload[k] = self.codec.decode(decoded) if success[k] else None
                error[k], num_symbols[k] = errors / size, num_re
                info[k] = {'mcs': mcs, 'resource_elements': num_re, 'tb_size': int(encoder.tb_size), 'bit_errors': errors}
        return success, payload, error, num_bits, num_symbols, info


class SionnaLinkChannel(RadioChannel):
    "5G NR link level with Sionna PHY: every bit LDPC-coded with a CRC, QAM, over the channel at each subcarrier."

    def __init__(
        self,
        codec: Codec | None = None, # Payload <-> bits; None: `IndexCodec(256)`
        fading: Fading | None = None, # The channel of a transmission (with subcarriers, e.g. `RayTracedPathsFading`)
        power_control: PowerControl | None = None, # Transmit power and outages; None: the link's tx_power_w
        noise_power: float = 1.0, # Noise power at the receiver, per resource element
        mcs_index: int | None = None, # A fixed MCS; None: link adaptation (see `SionnaErrorModel`)
        bler_target: float = 0.1, # Link adaptation's target error rate
        mcs_table_index: int = 1, # 5G NR MCS table: 1 (up to 64-QAM) or 2 (up to 256-QAM)
        mcs_category: int = 0, # 0: PUSCH (uplink), 1: PDSCH (downlink)
        num_bp_iter: int = 20, # LDPC decoder iterations
        seed: int | None = None, # Seed of the noise and the fading
        device: str | None = None, # Torch device; None: Sionna's default (a GPU if there is one)
        precision: str = 'single', # 'single' or 'double'
    ):
        self.link_level = SionnaLink(codec, mcs_index, bler_target, mcs_table_index, mcs_category, num_bp_iter, device, precision)
        self.sinr = SINR()
        super().__init__(fading, power_control, noise_power, seed)

    @property
    def codec(self) -> Codec:
        return self.link_level.codec

    @property
    def adaptation(self) -> SionnaErrorModel:
        "Link adaptation's PHY abstraction (Sionna SYS)."
        return self.link_level.adaptation

    @property
    def device(self) -> str:
        return self.link_level.device

    def receiver(self) -> list:
        "The SINR (link adaptation's), then the bits: `SionnaLink`."
        return [self.sinr, self.link_level]

In [ ]:
show_doc(SionnaLink.call)

### Tests of `SionnaLinkChannel`

These need Sionna, so they are flagged (`#| sionna`): `nbdev_test --flags sionna` runs them.

**The link level against the system level.** At MCS 10, 1000-bit messages are sent over a flat channel. The
fraction lost (the transport block error rate, TBLER) is compared with what `SionnaErrorModel` predicts. The PHY
abstraction should place the error-rate curve within about a dB of the simulation.

In [ ]:
from comm_core.dsp.source_coding import UniformQuantizer
from comm_core.fading import NoFading
from comm_core.power_control import ChannelInversion

In [ ]:
#| sionna
def send_all(channel, payloads, link=Link('a', 'b')):
    pairs = [(Transmission(message=Message('a', 'b', p), sender='a', receiver='b', start_time=0.0), link) for p in payloads]
    return channel.transmit_batch(pairs)

rng = np.random.default_rng(0)
codec = UniformQuantizer(bits=8, low=0, high=1)                        # 125 numbers: 1000 bits
payloads = [rng.uniform(0, 1, 125) for _ in range(200)]
model = SionnaErrorModel(mcs_index=10)
print(' SINR   simulated TBLER, 20 / 50 decoder iterations   SYS predicted')
for snr_db in (2.5, 3.0, 3.5, 4.0, 4.5, 5.0):
    lost = []
    for iterations in (20, 50):
        channel = SionnaLinkChannel(codec=codec, noise_power=10 ** (-snr_db / 10), mcs_index=10, num_bp_iter=iterations, seed=1)
        lost.append(1 - np.mean([r.success for r in send_all(channel, payloads)]))
    predicted = model([np.array([10 ** (snr_db / 10)])], np.array([1000]))[0][0]
    print(f'{snr_db:4.1f} dB {lost[0]:21.3f} / {lost[1]:.3f} {predicted:20.3f}')
    if predicted <= 0.01:
        assert lost[1] <= 0.2
    if predicted >= 0.99:
        assert lost[1] >= 0.8


 SINR   simulated TBLER, 20 / 50 decoder iterations   SYS predicted
 2.5 dB                 1.000 / 0.995                0.414
 3.0 dB                 0.980 / 0.765                0.120
 3.5 dB                 0.475 / 0.110                0.035
 4.0 dB                 0.055 / 0.010                0.009
 4.5 dB                 0.000 / 0.000                0.001
 5.0 dB                 0.000 / 0.000                0.000


* **With 50 decoder iterations,** the link level agrees with Sionna SYS where its tables have data (around 4 dB here).
* **With 20 iterations** (`TBDecoder`'s default, and so `num_bp_iter`'s), it needs about half a dB more.
* **Between table points,** the system level's curve is an interpolation. Its tables step 1.8 dB in SINR, so its
  waterfall is shallower than the real one.

**What arrives.** Codebook indices (49 of them, 8 bits each), with link adaptation: a delivered message is exactly
what was sent, and a lost one has failed its CRC.

In [ ]:
#| sionna
indices = np.arange(49) * 5
for snr_db in (0, 10, 25):
    channel = SionnaLinkChannel(noise_power=10 ** (-snr_db / 10), seed=0)          # IndexCodec(256), link adaptation
    rs = send_all(channel, [indices] * 20)
    ok = [r for r in rs if r.success]
    print(f'{snr_db:2d} dB: MCS {rs[0].metadata["mcs"]:2d}, {rs[0].metadata["resource_elements"]:4d} resource elements, '
          f'{len(ok)}/20 delivered')
    assert all((r.payload == indices).all() and r.metadata['bit_errors'] == 0 for r in ok)
    assert all(r.payload is None for r in rs if not r.success)
    assert rs[0].size_bits == 392
# reproducible with the seed; tiny payloads; an outage
a = [r.success for r in send_all(SionnaLinkChannel(noise_power=10 ** -0.35, mcs_index=10, seed=3), [indices] * 30)]
b = [r.success for r in send_all(SionnaLinkChannel(noise_power=10 ** -0.35, mcs_index=10, seed=3), [indices] * 30)]
assert a == b
r, = send_all(SionnaLinkChannel(noise_power=0.01, mcs_index=10, seed=0), [np.array([7, 200])])   # 16 bits
assert r.success and r.payload.tolist() == [7, 200]
r, = send_all(SionnaLinkChannel(fading=NoFading(0.1), power_control=ChannelInversion(10, max_power=10)), [indices])
assert not r.success and r.metadata['outage']
try:
    SionnaLinkChannel(mcs_index=0); raise AssertionError('no error')        # code rate 0.12: not encodable
except ValueError as e:
    assert '1/5' in str(e)
assert min(SionnaLinkChannel().adaptation.mcs.tolist()) == 3

 0 dB: MCS  5,  530 resource elements, 17/20 delivered
10 dB: MCS 17,  153 resource elements, 16/20 delivered
25 dB: MCS 27,   75 resource elements, 20/20 delivered


**Small messages: where Sionna SYS is wrong.** Link adaptation targets 10% losses using Sionna SYS's predictions.
At the link level, messages of a few hundred bits or more meet the target. Small messages are lost far more
often: Sionna's tables were simulated for code blocks from 24 bits, at coarse SINR steps, and are much too
optimistic for small transport blocks at high code rates. For small messages, use a fixed, lower `mcs_index`, or
check the MCS at the link level.

In [ ]:
#| sionna
rng = np.random.default_rng(0)
print('bits  SINR  MCS  REs  lost (target 0.1)')
lost = {}
for numbers in (2, 8, 49, 125):
    for snr_db in (5, 15, 25):
        channel = SionnaLinkChannel(codec=UniformQuantizer(8, 0, 1), noise_power=10 ** (-snr_db / 10), num_bp_iter=50, seed=0)
        rs = send_all(channel, [rng.uniform(0, 1, numbers) for _ in range(100)])
        lost[8 * numbers, snr_db] = 1 - np.mean([r.success for r in rs])
        print(f'{8 * numbers:4d} {snr_db:5d} {rs[0].metadata["mcs"]:4d} {rs[0].metadata["resource_elements"]:4d}  {lost[8 * numbers, snr_db]:.2f}')
assert all(lost[392, s] <= 0.2 for s in (5, 15, 25))                   # a few hundred bits: about the target

bits  SINR  MCS  REs  lost (target 0.1)
  16     5   10   13  1.00
  16    15   21    8  0.80
  16    25   26   17  1.00
  64     5   11   44  0.55
  64    15   21   18  0.35
  64    25   26   17  1.00
 392     5   11  266  0.02
 392    15   22  101  0.02
 392    25   27   75  0.00
1000     5   12  590  0.24
1000    15   22  257  0.00
1000    25   27  188  0.00


**In a network**, over a channel ray-traced at anchors with 52 subcarriers (here line-of-sight paths along a
corridor). Four nodes take turns (`TDMAProtocol`) and share the medium (`InterferenceBackend`):

In [ ]:
from comm_core import Network, Node, FullMeshTopology, TDMAProtocol, InterferenceBackend, RayTracedPathsFading, grid_points

In [ ]:
#| sionna
C, F = 299_792_458.0, 5.18e9
points = grid_points((0, 0), (30, 0), 0.5, height=1.0)
v = points[None] - points[:, None]
L = np.linalg.norm(v, axis=-1)
same = L == 0
L = np.where(same, 1.0, L)
paths = dict(a=np.where(same, 0, C / F / (4 * np.pi * L) * np.exp(-2j * np.pi * F * L / C))[..., None],
             tau=(L / C)[..., None], u_tx=(v / L[..., None])[..., None, :], u_rx=(-v / L[..., None])[..., None, :])
fading = RayTracedPathsFading(points, points, **paths, frequency=F, subcarriers=(np.arange(52) - 25.5) * 312.5e3)
nodes = ['a', 'b', 'c', 'd']
fading.set_positions({'a': (0.3, 0, 1), 'b': (6.1, 0, 1), 'c': (14.8, 0, 1), 'd': (29.6, 0, 1)})
channel = SionnaLinkChannel(fading=fading, noise_power=10 ** ((-94 - 30) / 10), seed=0)
net = Network([Node(n) for n in nodes], FullMeshTopology(nodes), TDMAProtocol(nodes, queue_size=3), InterferenceBackend(channel))
for link in (net.topology.get_link(s, r) for s in nodes for r in nodes if s != r):
    link.tx_power_w, link.bandwidth_hz = 1e-4, 20e6                     # 0.1 mW, 20 MHz
indices = np.arange(49) * 5
for step in range(4):
    for s in nodes:
        for r in nodes:
            if s != r:
                net.send(Message(s, r, indices))
    for res in net.step():
        t = res.transmission
        print(f'{t.sender} -> {t.receiver}: SINR {res.snr_db:5.1f} dB, MCS {res.metadata["mcs"]:2d}, '
              f'{res.metadata["resource_elements"]:4d} REs, delivered {res.success}')
received = net.receive('a')
assert net.metrics.summary().transmissions == 12 and all((m.payload == indices).all() for m in received)

a -> b: SINR  22.0 dB, MCS 27,   75 REs, delivered True
a -> c: SINR  14.0 dB, MCS 21,  109 REs, delivered True
a -> d: SINR   7.9 dB, MCS 14,  182 REs, delivered True
b -> a: SINR  22.0 dB, MCS 27,   75 REs, delivered True
b -> c: SINR  18.5 dB, MCS 25,   82 REs, delivered True
b -> d: SINR   9.8 dB, MCS 17,  153 REs, delivered False
c -> a: SINR  14.0 dB, MCS 21,  109 REs, delivered True
c -> b: SINR  18.5 dB, MCS 25,   82 REs, delivered True
c -> d: SINR  13.9 dB, MCS 21,  109 REs, delivered True
d -> a: SINR   7.9 dB, MCS 14,  182 REs, delivered True
d -> b: SINR   9.8 dB, MCS 17,  153 REs, delivered False
d -> c: SINR  13.9 dB, MCS 21,  109 REs, delivered True


A broadcast: one MCS for all its receivers, the weakest one's (see `SionnaErrorModel`).

In [ ]:
#| sionna
net.reset()
for r in ('b', 'c', 'd'):
    net.send(Message('a', r, indices))
unicast = {res.transmission.receiver: res.metadata['mcs'] for res in net.step()}
net.reset()
net.broadcast('a', indices)
results = net.step()
broadcast = {res.transmission.receiver: res.metadata['mcs'] for res in results}
print('unicast MCSs:', unicast, ' broadcast:', broadcast, ' delivered:', [res.success for res in results])
assert len(set(unicast.values())) > 1 and set(broadcast.values()) == {min(unicast.values())}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()